# BTL Hệ CSDL đa phương tiện - Nhóm 19: tìm kiếm ảnh chân dung đàn ông

Chạy lần lượt từ trên xuống. Mã nguồn nằm ở 7 tệp:

| Tệp | Nội dung |
|---|---|
| `xu_ly_anh.py` | các phép xử lý tự code bằng numpy (ảnh xám, Laplace, căn chỉnh + nội suy, tô đa giác, RGB→HSV, LBP, Otsu) |
| `tien_xu_ly.py` | tiền xử lý 6 bước B1 → B6; làm vuông ảnh truy vấn chưa đúng 256×256 |
| `trich_rut.py` | vector đặc trưng 39 chiều |
| `csdl.py` | CSDL SQLite 2 bảng: tạo bảng, nạp dữ liệu, kiểm tra toàn vẹn, thêm ảnh mới |
| `tim_kiem.py` | chuẩn hóa min-max, khoảng cách, top-5 (đọc dữ liệu từ CSDL) |
| `danh_gia.py` | phép thử ảnh biến đổi (đánh giá kết quả) |
| `giao_dien.py` | giao diện demo Tkinter |

Hình xuất ra thư mục `figs/` để chèn vào báo cáo.

In [ ]:
import os, csv
import numpy as np
import matplotlib.pyplot as plt

import xu_ly_anh as XL
import tien_xu_ly as TXL
import trich_rut as TR
import tim_kiem as TK

os.makedirs("figs", exist_ok=True)
MA_MAU = "man_0097"        # ảnh dùng để minh họa các bước


def doc_log():
    with open(XL.TEP_NHAT_KY, newline="") as f:
        return list(csv.DictReader(f))


def hien(a, anh, tieu_de=""):
    """Vẽ một ảnh đã căn chỉnh, chỉ hiện vùng quanh khuôn mặt (mặt luôn nằm ở cùng vị trí)."""
    a.imshow(anh); a.set_xlim(38, 218); a.set_ylim(236, 26); a.axis("off")
    a.set_title(tieu_de, fontsize=9)


def bang(tieu_de, hang):
    """In một bảng số liệu dạng văn bản."""
    print(" | ".join(tieu_de))
    for h in hang:
        print(" | ".join(f"{x:.4f}" if isinstance(x, float) else str(x) for x in h))

## 0. Đánh số ảnh (chỉ chạy 1 lần)
Chép ảnh thu thập được vào `data/anh256/`, đặt tên `man_0001.jpg`…; bảng đối chiếu tên gốc ở `data/ten_goc.csv`.

In [ ]:
# TXL.danh_so_anh("1000PictureFinal")      # bỏ dấu # nếu cần đánh số lại từ thư mục ảnh gốc
print("So anh tho:", len(os.listdir(XL.THU_MUC_ANH_GOC)))

## 1. Tiền xử lý (6 bước B1 → B6)
Mất khoảng 4 phút cho ~1000 ảnh. Kết quả lưu ở `data/tien_xu_ly/` (ảnh đã xử lý lưu `.jpg` để xem; đặc trưng tính trực tiếp, không đọc lại từ các tệp này).

In [ ]:
TXL.chay()

### Số ảnh qua tiền xử lý và thống kê trên các ảnh đạt

In [ ]:
log = doc_log()
dat = [r for r in log if r["dat"] == "1"]
print("Tong:", len(log), "| Dat:", len(dat))
ly_do = {}
for r in log:
    if r["dat"] != "1":
        ly_do[r["ly_do"]] = ly_do.get(r["ly_do"], 0) + 1
print("Ly do loai:", ly_do)
print("So anh dung cua so diem SVM < 0 (mat yeu):", sum(float(r["diem_mat"]) < 0 for r in dat))
print()
ten = {"goc_nghieng": "Goc nghieng truoc can chinh (do)", "do_net": "Do net (phuong sai Laplace)",
       "ti_le_khung": "Ti le khung mat dlib / anh (anh tho)", "lech_tam": "Lech tam khung mat (px, anh tho)",
       "he_so": "He so co gian s", "ti_le_mat": "Ti le mat na mat / anh (sau can chinh)",
       "lech_moc": "Do lech diem moc 2 lan do", "thieu": "Phan mat na thieu du lieu"}
hang = []
for k, t in ten.items():
    v = np.array([float(r[k]) for r in dat])
    hang.append([t, float(v.min()), float(v.max()), float(v.mean())])
bang(["Chi so", "Nho nhat", "Lon nhat", "Trung binh"], hang)

### Hình 1.3 - Sơ đồ 6 bước tiền xử lý; Hình 2.1 - Sơ đồ trích rút đặc trưng

In [ ]:
from matplotlib.patches import FancyBboxPatch

def hop(a, x, y, w, h, chu, mau="#eeeeee", co=10):
    a.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.06",
                               fc=mau, ec="#333333", lw=1.3))
    a.text(x + w / 2, y + h / 2, chu, ha="center", va="center", fontsize=co)

def mui_ten(a, x1, y1, x2, y2):
    a.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", lw=1.4, color="#333333"))

XANH, VANG, LUC = "#dce9f7", "#fff1cc", "#ddf0dd"

# Hình 1.3 - sơ đồ 6 bước tiền xử lý
hinh_ve, a = plt.subplots(figsize=(15, 3.2)); a.set_xlim(0, 15); a.set_ylim(0, 3.2); a.axis("off")
buoc = [("Ảnh gốc\n256×256", "#eeeeee"), ("1. Đọc ảnh,\nkiểm tra quy cách", XANH), ("2. Chuyển\nảnh xám", XANH),
        ("3. Kiểm tra\nđộ nét", XANH), ("4. Phát hiện mặt\n+ 68 điểm mốc", VANG), ("5. Căn chỉnh\ntheo hai mắt", XANH),
        ("6. Lọc nền", XANH), ("Ảnh đã xử lý\n+ 68 điểm mốc", LUC)]
w, kc = 1.55, 1.85
for i, (chu, mau) in enumerate(buoc):
    hop(a, 0.15 + i * kc, 1.1, w, 1.3, chu, mau)
    if i:
        mui_ten(a, 0.15 + (i - 1) * kc + w, 1.75, 0.15 + i * kc, 1.75)
a.text(0.15, 0.45, "Vàng: dùng thư viện dlib.   Xanh: nhóm tự viết.   Ảnh không đạt ở bước 1, 3 hoặc 4 bị loại.",
       fontsize=10, color="#444444")
plt.tight_layout(); plt.savefig("figs/fig_flow_pre.png", dpi=150); plt.show()

# Hình 2.1 - sơ đồ trích rút đặc trưng
hinh_ve, a = plt.subplots(figsize=(15, 6)); a.set_xlim(0, 15); a.set_ylim(0, 6); a.axis("off")
hop(a, 0.2, 2.4, 2.1, 1.2, "Ảnh gốc\n256×256", co=12)
hop(a, 3.0, 2.3, 2.3, 1.4, "Tiền xử lý\n6 bước\n(Phần 1)", XANH, co=12)
giua = [(4.5, "Ảnh màu đã căn chỉnh,\nđã lọc nền"), (2.5, "68 điểm mốc\nsau căn chỉnh"), (0.5, "Ảnh xám\nđã căn chỉnh")]
bo = [(4.5, "Bộ màu sắc: lược đồ\nH × S trên vùng da\n32 chiều"), (2.5, "Bộ hình học: 5 tỉ số\ngiữa các điểm mốc\n5 chiều"),
      (0.5, "Bộ kết cấu: LBP vùng\ncằm – miệng, lông mày\n2 chiều")]
mui_ten(a, 2.3, 3.0, 3.0, 3.0)
for y, chu in giua:
    hop(a, 6.1, y, 2.5, 1.0, chu)
    mui_ten(a, 5.3, 3.0, 6.1, y + 0.5)
for y, chu in bo:
    hop(a, 9.4, y, 2.8, 1.0, chu, XANH)
    mui_ten(a, 12.2, y + 0.5, 12.9, 3.0)
mui_ten(a, 8.6, 5.0, 9.4, 5.0); mui_ten(a, 8.6, 1.0, 9.4, 1.0)
for y in (5.0, 3.0, 1.0):
    mui_ten(a, 8.6, 3.0, 9.4, y)
hop(a, 12.9, 2.4, 1.9, 1.2, "Vector\n39 chiều\n→ CSDL", LUC, co=12)
plt.tight_layout(); plt.savefig("figs/fig_flow_feat.png", dpi=150); plt.show()

### Hình 1.4 - Một ảnh đi qua các bước

In [ ]:
rgb0 = TXL.doc_rgb(os.path.join(XL.THU_MUC_ANH_GOC, MA_MAU + ".jpg"))                 # B1
xam0 = XL.chuyen_xam(rgb0)                                                     # B2
print("Do net:", round(XL.phuong_sai_laplace(xam0), 1))                          # B3
so_mat, khung, moc0, diem = TXL.tim_diem_moc(np.round(xam0).astype(np.uint8))   # B4
trai, phai = XL.tam_hai_mat(moc0)                                            # B5
theta = np.arctan2(phai[1] - trai[1], phai[0] - trai[0])
s = XL.D_MAT / XL.khoang_cach_diem(trai, phai)
m = (trai + phai) / 2
print(f"theta = {np.degrees(theta):.2f} do, s = {s:.3f}, trung diem hai mat = ({m[0]:.1f}; {m[1]:.1f})")
rgb1 = np.clip(np.round(XL.can_chinh_anh(rgb0, theta, s, m)), 0, 255).astype(np.uint8)
moc1 = XL.can_chinh_diem(moc0, theta, s, m)
rgb2 = rgb1.copy()                                                          # B6
rgb2[~TXL.mat_na_khuon_mat(moc1)] = 255

hinh_ve, truc = plt.subplots(1, 5, figsize=(15, 3.4))
truc[0].imshow(rgb0); truc[0].set_title("1. Ảnh gốc 256×256")
truc[1].imshow(xam0, cmap="gray", vmin=0, vmax=255); truc[1].set_title("2. Ảnh xám")
truc[2].imshow(xam0, cmap="gray", vmin=0, vmax=255); truc[2].set_title("4. Khung mặt + 68 điểm mốc")
l, t, r, b = khung
truc[2].plot([l, r, r, l, l], [t, t, b, b, t], "-", color="yellow", lw=1.5)
truc[2].plot(moc0[:, 0], moc0[:, 1], ".", color="lime", ms=4)
truc[3].imshow(rgb1); truc[3].set_title("5. Căn chỉnh theo hai mắt")
truc[3].plot(moc1[:, 0], moc1[:, 1], ".", color="lime", ms=4)
truc[3].axhline(XL.MAT_Y, color="yellow", lw=0.8, ls="--")
truc[4].imshow(rgb2); truc[4].set_title("6. Lọc nền")
for a in truc:
    a.set_xlim(0, 255); a.set_ylim(255, 0); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig("figs/fig_steps.png", dpi=150); plt.show()

### Hình 1.6 - Kết quả tiền xử lý của 30 ảnh lấy đều trong bộ dữ liệu

In [ ]:
# 30 ảnh minh họa, chọn những ảnh thấy rõ trán và lông mày
chon = ["man_" + s for s in "0361 0064 0155 0423 0194 0498 0218 0495 0117 0344 0682 0041 0360 0367 0813 0447 0970 0793 0914 0899 0010 0005 0158 0141 0083 0989 0197 0050 0558 0506".split()]
hinh_ve, truc = plt.subplots(5, 6, figsize=(12, 11))
for a, ma in zip(truc.ravel(), chon):
    a.imshow(TR.doc_anh_da_xu_ly(ma)[0]); a.set_xlim(38, 218); a.set_ylim(236, 26); a.axis("off")
plt.tight_layout(); plt.savefig("figs/fig_prep30.png", dpi=110); plt.show()

### Hình 1.1 - Một số ảnh trong bộ dữ liệu (ảnh gốc 256×256)

In [ ]:
# 12 ảnh minh họa, chọn những ảnh thấy rõ trán và lông mày
chon = ["man_" + s for s in "0107 0739 0973 0166 0816 0943 0388 0483 0372 0396 0125 0570".split()]
hinh_ve, truc = plt.subplots(2, 6, figsize=(13, 4.5))
for a, ma in zip(truc.ravel(), chon):
    a.imshow(TXL.doc_rgb(os.path.join(XL.THU_MUC_ANH_GOC, ma + ".jpg"))); a.axis("off")
plt.tight_layout(); plt.savefig("figs/fig_dataset.png", dpi=130); plt.show()

## 2. Trích rút đặc trưng (39 chiều)

In [ ]:
TR.chay()

### Tạo CSDL SQLite (2 bảng) và kiểm tra toàn vẹn
`TK.tinh_min_max` tính min, max từng chiều trên toàn bộ CSDL (lưu `data/min_max.json`); `CSDL.tao_csdl` tạo 2 bảng
`anh`, `dac_trung` (quan hệ 1 – 1 qua khóa ngoại `ma_anh`) và nạp dữ liệu trong một giao dịch; `CSDL.kiem_tra` chạy các câu lệnh kiểm tra.
Từ đây mọi bước đọc vector từ tệp `data/khuon_mat.db`; min, max đọc từ `min_max.json`, trọng số 1/3 là hằng số `TK.TRONG_SO`.

In [ ]:
import csdl as CSDL
TK.tinh_min_max(CSDL.doc_tep_dac_trung()[1])     # min, max từng chiều -> data/min_max.json
CSDL.tao_csdl()                            # tạo 2 bảng và nạp dữ liệu
CSDL.kiem_tra()                            # kiểm tra toàn vẹn
print(CSDL.DDL)                            # lệnh tạo bảng

### Hình 1.5 - 68 điểm mốc; Hình 2.2 - các vùng lấy đặc trưng kết cấu và màu

In [ ]:
ds_moc = TR.doc_diem_moc()
rgb, xam = TR.doc_anh_da_xu_ly(MA_MAU)
moc = ds_moc[MA_MAU]

hinh_ve, truc = plt.subplots(figsize=(7.5, 7.5))
truc.imshow(xam, cmap="gray", vmin=0, vmax=255, alpha=0.55)
nhom = [(0, 17, "tab:green", "viền hàm 0-16"), (17, 27, "tab:orange", "lông mày 17-26"),
        (27, 36, "tab:blue", "mũi 27-35"), (36, 48, "tab:red", "mắt 36-47"), (48, 68, "tab:purple", "miệng 48-67")]
for a, b, mau, ten_nhom in nhom:
    truc.plot(moc[a:b, 0], moc[a:b, 1], "o", color=mau, ms=4, label=ten_nhom)
    for i in range(a, b):
        truc.text(moc[i, 0] + 1.2, moc[i, 1] - 1.2, str(i), fontsize=7, color=mau)
x1, x2 = moc[:, 0].min() - 12, moc[:, 0].max() + 12
y1, y2 = moc[:, 1].min() - 22, moc[:, 1].max() + 10
truc.set_xlim(x1, x2); truc.set_ylim(y2, y1); truc.set_xticks([]); truc.set_yticks([])
truc.legend(loc="upper center", ncol=3, fontsize=8)
plt.tight_layout(); plt.savefig("figs/fig_landmarks.png", dpi=150); plt.show()

hinh_ve, truc = plt.subplots(1, 3, figsize=(12, 4.2))
to = np.zeros((XL.CANH_ANH, XL.CANH_ANH, 4))
to[TR.vung_cam(moc)] = (1, 0, 0, 0.45)
for v in TR.vung_long_may(moc):
    to[v] = (0, 0.4, 1, 0.5)
truc[0].imshow(xam, cmap="gray", vmin=0, vmax=255); truc[0].imshow(to)
truc[0].set_title("Kết cấu: cằm - miệng (đỏ), lông mày (xanh)")
truc[1].imshow(XL.ma_lbp(xam), cmap="gray"); truc[1].set_title("Ảnh mã LBP (0..255)")
to = np.zeros((XL.CANH_ANH, XL.CANH_ANH, 4)); to[TR.vung_da(moc)] = (0, 1, 0, 0.45)
truc[2].imshow(rgb); truc[2].imshow(to); truc[2].set_title("Màu: mặt nạ da (2 má + trán)")
for a in truc:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig("figs/fig_regions.png", dpi=150); plt.show()

### Giá trị đặc trưng của ảnh mẫu và thống kê trên cả bộ dữ liệu

In [ ]:
ds_ma, X = TK.doc_vector_csdl()
v = X[ds_ma.index(MA_MAU)]
print(MA_MAU, "- 7 dac trung dau:")
for ten_cot, gia_tri in zip(TR.TEN_COT[:7], v[:7]):
    print(f"  {ten_cot:14s} = {gia_tri:.4f}")
print("  luoc do H x S (hang = o H 0..7, cot = o S 0..3):")
print(np.round(v[7:].reshape(TR.SO_O_H, TR.SO_O_S), 3))
print()
bang(["Dac trung", "Nho nhat", "Lon nhat", "Trung binh", "Do lech chuan"],
     [[n, float(X[:, j].min()), float(X[:, j].max()), float(X[:, j].mean()), float(X[:, j].std())]
      for j, n in enumerate(TR.TEN_COT)])

### Thống kê sắc độ H của các điểm da (lý do chia ô H trong 0°-40°)

In [ ]:
dem = np.zeros(4)            # H < 40 | H >= 330 | 40 <= H < 330 | tổng
tb_luoc_do = X[:, 7:].mean(axis=0).reshape(TR.SO_O_H, TR.SO_O_S)
for ma in ds_ma:               # tính trên kết quả tiền xử lý trực tiếp (như lúc trích đặc trưng)
    kq_i = TXL.xu_ly_mot_anh(os.path.join(XL.THU_MUC_ANH_GOC, ma + ".jpg"))
    da = TR.vung_da(kq_i["moc"]) & ~(kq_i["rgb"] == 255).all(axis=2)
    h, s_, _ = XL.rgb_sang_hsv(kq_i["rgb"][da])
    dem += [(h < 40).sum(), (h >= 330).sum(), ((h >= 40) & (h < 330)).sum(), h.size]
print(f"So diem da trung binh moi anh: {dem[3] / len(ds_ma):.0f}")
print(f"H < 40 do: {100 * dem[0] / dem[3]:.1f} % | H >= 330 do: {100 * dem[1] / dem[3]:.1f} % | 40 <= H < 330: {100 * dem[2] / dem[3]:.1f} %")
print("Ti le trung binh theo o H (0..7):", np.round(tb_luoc_do.sum(axis=1), 3))
print("Ti le trung binh theo o S (0..3):", np.round(tb_luoc_do.sum(axis=0), 3))

### Hình 1.2 - Ví dụ khuôn mặt giống nhau về dáng mặt nhưng khác nhau ở các điểm khác (ảnh gốc, xếp theo `hh_khung_mat`)

In [ ]:
def vien_trang(anh, rong=6, nguong=245):
    """True nếu một cạnh ảnh gần như toàn trắng (ảnh được đệm viền hoặc nền trắng sát mép)."""
    trang = anh.min(axis=2) >= nguong
    return max(trang[:rong].mean(), trang[-rong:].mean(), trang[:, :rong].mean(), trang[:, -rong:].mean()) > 0.97


def chon_quanh(vi_tri, thu_tu, k=5):
    """Lấy k ảnh không có viền trắng, gần vị trí vi_tri nhất trong thứ tự tăng dần của hh_khung_mat."""
    chon, buoc = [], 0
    while len(chon) < k:
        for j in sorted({vi_tri - buoc, vi_tri + buoc}):
            if 0 <= j < len(thu_tu) and len(chon) < k:
                i = thu_tu[j]
                if ds_ma[i] in TRAN_THOANG and not vien_trang(TXL.doc_rgb(os.path.join(XL.THU_MUC_ANH_GOC, ds_ma[i] + ".jpg"))):
                    chon.append(i)
        buoc += 1
    return chon


# chỉ lấy ảnh thấy rõ trán và lông mày để minh họa
TRAN_THOANG = {"man_" + s for s in "0339 0494 0295 0513 0567 0828 0224 0921 0078 0196 0281 0035 0779 0273 0327".split()}
thu_tu = np.argsort(X[:, 0])
n = len(ds_ma)
nhom = [("Dáng mặt dài", chon_quanh(n // 10, thu_tu)),
        ("Dáng mặt trung bình", chon_quanh(n // 2, thu_tu)),
        ("Dáng mặt rộng", chon_quanh(9 * n // 10, thu_tu))]
hinh_ve, truc = plt.subplots(3, 5, figsize=(10, 6.9))
for hang, (ten_nhom, chi_so) in enumerate(nhom):
    for cot, i in enumerate(chi_so):
        truc[hang, cot].imshow(TXL.doc_rgb(os.path.join(XL.THU_MUC_ANH_GOC, ds_ma[i] + ".jpg"))); truc[hang, cot].axis("off")
    truc[hang, 2].set_title(ten_nhom, fontsize=11)
plt.tight_layout(); plt.savefig("figs/fig_similar.png", dpi=130); plt.show()

## 3. Chuẩn hóa và tìm kiếm top-5
min, max của từng chiều đọc từ `data/min_max.json` (`TK.doc_min_max`).

In [ ]:
mn, mx = TK.doc_min_max()                   # min, max từ min_max.json
Xn = TK.chuan_hoa(X, mn, mx)
print("Sau chuan hoa: nho nhat", Xn.min(), "| lon nhat", Xn.max())


def hien_ket_qua(anh_truy_van, top, ten_tep):
    hinh_ve, truc = plt.subplots(1, 6, figsize=(13, 2.9))
    hien(truc[0], anh_truy_van, "Ảnh truy vấn")
    for a, r in zip(truc[1:], top):
        hien(a, TR.doc_anh_da_xu_ly(r["ma_anh"])[0], f"{r['ma_anh']}\nS = {r['S']:.4f}")
    plt.tight_layout(); plt.savefig(ten_tep, dpi=130); plt.show()


# Kiểm tra: lấy chính một ảnh trong CSDL làm truy vấn -> ảnh đó phải đứng đầu với S = 1
kq, v, top = TK.tim_theo_anh(os.path.join(XL.THU_MUC_ANH_GOC, MA_MAU + ".jpg"))
bang(["Anh", "S", "D", "D hinh hoc", "D ket cau", "D mau sac"],
     [[r["ma_anh"], r["S"], r["D"], r["D_hinh_hoc"], r["D_ket_cau"], r["D_mau_sac"]] for r in top])
hien_ket_qua(kq["rgb"], top, "figs/fig_top5.png")

### Truy vấn như một ảnh mới
Mỗi người chỉ có 1 ảnh nên để thử "ảnh mới", lấy một ảnh trong CSDL làm truy vấn và **không cho chính nó vào kết quả** (`bo_qua`).
In kèm kết quả trung gian: vector gốc, vector chuẩn hóa, khoảng cách từng bộ.
Với ảnh nằm ngoài CSDL: `kq, v, top = TK.tim_theo_anh("duong_dan.jpg")`.

In [ ]:
MA_TRUY_VAN = "man_0050"
v = X[ds_ma.index(MA_TRUY_VAN)]
q = TK.chuan_hoa(v, *TK.doc_min_max())
top = TK.tim_theo_vector(v, bo_qua=MA_TRUY_VAN)
print("7 dac trung dau (goc)      :", np.round(v[:7], 4))
print("7 dac trung dau (chuan hoa):", np.round(q[:7], 4))
bang(["Anh", "S", "D", "D hinh hoc", "D ket cau", "D mau sac"],
     [[r["ma_anh"], r["S"], r["D"], r["D_hinh_hoc"], r["D_ket_cau"], r["D_mau_sac"]] for r in top])
hien_ket_qua(TR.doc_anh_da_xu_ly(MA_TRUY_VAN)[0], top, "figs/fig_query.png")

## 4. Báo cáo 2: hình và số liệu
### Hình 3.1 - Lược đồ CSDL (2 bảng)

In [ ]:
from matplotlib.patches import FancyBboxPatch

def bang_erd(a, x, y, w, ten, cot, mau):
    """Vẽ một bảng: tiêu đề + các dòng (khóa, tên cột, kiểu). Trả về chiều cao."""
    h = 0.42 * (len(cot) + 1)
    a.add_patch(FancyBboxPatch((x, y - h), w, h, boxstyle="round,pad=0.02", fc="white", ec="#333333", lw=1.4))
    a.add_patch(FancyBboxPatch((x, y - 0.42), w, 0.42, boxstyle="round,pad=0.02", fc=mau, ec="#333333", lw=1.4))
    a.text(x + w / 2, y - 0.21, ten, ha="center", va="center", fontsize=12, weight="bold", color="white")
    for k, (khoa, ten_cot, kieu) in enumerate(cot):
        yy = y - 0.42 * (k + 1.5)
        a.text(x + 0.1, yy, khoa, fontsize=8.5, va="center", weight="bold",
               color="#b00000" if khoa.startswith("PK") else ("#0060c0" if khoa else "#555555"))
        a.text(x + 0.85, yy, ten_cot, fontsize=9.5, va="center")
        a.text(x + w - 0.1, yy, kieu, fontsize=8.5, va="center", ha="right", color="#666666")
    return h

# Hình 3.x - ERD: 2 bảng, anh 1 - 1 dac_trung qua khóa ngoại ma_anh
hinh_ve, a = plt.subplots(figsize=(10, 4.9)); a.set_xlim(0, 10); a.set_ylim(0, 4.9); a.axis("off")
bang_erd(a, 0.3, 4.6, 3.7, "anh", [("PK", "ma_anh", "TEXT"), ("UQ", "duong_dan", "TEXT"), ("", "rong", "INTEGER"),
         ("", "cao", "INTEGER"), ("", "dinh_dang", "TEXT"), ("", "ngay_nap", "TEXT")], "#4a6fa5")
bang_erd(a, 5.8, 4.6, 3.9, "dac_trung", [("PK, FK", "ma_anh", "TEXT"), ("", "hh_khung_mat (X0)", "REAL"),
         ("", "...   hh_cao_mui (X4)", "REAL"), ("", "kc_lbp_cam (X5)", "REAL"), ("", "kc_long_may (X6)", "REAL"),
         ("", "ms_h0_s0 (X7)", "REAL"), ("", "...   ms_h7_s3 (X38)", "REAL")], "#3c8d5a")
a.plot([4.0, 5.8], [3.97, 3.97], color="k", lw=1.6)
a.text(4.1, 4.08, "1", fontsize=11, weight="bold"); a.text(5.62, 4.08, "1", fontsize=11, weight="bold")
a.text(4.9, 3.72, "ma_anh", ha="center", fontsize=9, color="#0060c0")
a.text(0.3, 1.05, "PK: khóa chính   FK: khóa ngoại   UQ: duy nhất (UNIQUE)\n"
       "anh – dac_trung: quan hệ 1 – 1, khóa chính ma_anh của dac_trung đồng thời là khóa ngoại tham chiếu anh(ma_anh)\n"
       "anh lưu đặc trưng tổng quát, dac_trung lưu 39 đặc trưng nội dung (giá trị gốc)", fontsize=9.5, va="top", color="#333333")
plt.tight_layout(); plt.savefig("figs/fig_erd.png", dpi=150); plt.show()


### Hình 4.1 - Sơ đồ khối hệ thống (luồng nạp CSDL và luồng truy vấn)

In [ ]:
hinh_ve, a = plt.subplots(figsize=(16, 7.2)); a.set_xlim(0, 16); a.set_ylim(0, 7.2); a.axis("off")
w, h = 1.75, 1.15
nap = [("Bộ dữ liệu\n1005 ảnh JPG\n256×256", "#eeeeee"), ("Tiền xử lý\n6 bước", XANH), ("Trích rút\n39 đặc trưng", XANH),
       ("Tính min, max\ntừng chiều", XANH), ("CSDL SQLite\n2 bảng", LUC)]
ra_nap = ["ảnh RGB", "ảnh đã căn chỉnh,\nlọc nền + 68 điểm", "vector gốc\n(dac_trung.csv)", "vector gốc\n(nạp bảng dac_trung)"]
x_nap = [0.2, 2.75, 5.3, 7.85, 10.4]
for k, ((chu, mau), x) in enumerate(zip(nap, x_nap)):
    hop(a, x, 5.3, w, h, chu, mau, co=10.5)
    if k:
        mui_ten(a, x_nap[k - 1] + w, 5.875, x, 5.875)
        a.text((x_nap[k - 1] + w + x) / 2, 6.6, ra_nap[k - 1], ha="center", va="center", fontsize=8.5, color="#444444")
a.text(0.2, 6.95, "Luồng 1 – Nạp CSDL (chạy một lần)", fontsize=11.5, weight="bold")

tv = [("Giao diện:\nnhận ảnh JPG,\nlàm vuông 256×256", "#eeeeee"), ("Tiền xử lý\n6 bước", XANH), ("Trích rút\n39 đặc trưng", XANH),
      ("Chuẩn hóa\nmin-max", XANH), ("So khớp:\nD từng bộ, D, S", XANH), ("Xếp hạng:\nS giảm dần,\nlấy 5 ảnh", XANH),
      ("Hiển thị top-5\n+ giải thích", "#eeeeee")]
ra_tv = ["ảnh\n256×256", "ảnh đã căn chỉnh,\nlọc nền + 68 điểm", "vector q\n(39 giá trị)", "q'\n(trong [0; 1])", "S của\n1005 ảnh", "5 mã ảnh,\nS, D"]
x_tv = [0.2 + k * 2.27 for k in range(7)]
for k, ((chu, mau), x) in enumerate(zip(tv, x_tv)):
    hop(a, x, 1.2, w, h + 0.15, chu, mau, co=10)
    if k:
        mui_ten(a, x_tv[k - 1] + w, 1.85, x, 1.85)
        a.text((x_tv[k - 1] + w + x) / 2, 0.75, ra_tv[k - 1], ha="center", va="center", fontsize=8.5, color="#444444")
a.text(0.2, 3.0, "Luồng 2 – Truy vấn (mỗi lần nhập một ảnh)", fontsize=11.5, weight="bold")
mui_ten(a, x_nap[3] + w / 2, 5.3, x_tv[3] + w / 2, 2.6)       # min, max lấy từ min_max.json
a.text(8.9, 4.15, "min, max\n(min_max.json)", ha="left", fontsize=8.5, color="#444444")
mui_ten(a, 11.7, 5.3, x_tv[4] + w / 2, 2.6)
a.text(12.45, 4.15, "1005 vector\n(bảng dac_trung)", ha="left", fontsize=8.5, color="#444444")
plt.tight_layout(); plt.savefig("figs/fig_so_do_khoi.png", dpi=150); plt.show()

### Kết quả trung gian của một truy vấn (mục 4b)
Ảnh truy vấn là một biến thể của ảnh mẫu `MA_MAU` (sáng hơn, hơi nghiêng, lệch khung, thu nhỏ, nén lại), coi như một ảnh mới
của người đã có trong CSDL. Ảnh được lưu ở `data/truy_van_mau/` để dùng lại khi demo.

In [ ]:
import danh_gia as DG
os.makedirs("data/truy_van_mau", exist_ok=True)
TEP_MAU = f"data/truy_van_mau/{MA_MAU}_bien_the.jpg"
anh, chat_luong = DG.bien_doi(TXL.doc_rgb(os.path.join(XL.THU_MUC_ANH_GOC, MA_MAU + ".jpg")), "tong_hop")
TXL.cv2.imwrite(TEP_MAU, anh[:, :, ::-1], [TXL.cv2.IMWRITE_JPEG_QUALITY, chat_luong])

kq = TXL.xu_ly_mot_anh(TEP_MAU)
v = TR.trich_vector(kq["rgb"], kq["xam"], kq["moc"])
xh = TK.xep_hang(v)
print(f"Tien xu ly: do net {kq['do_net']:.1f}, lech moc {kq['lech_moc']:.3f}, goc nghieng {kq['goc_nghieng']:.2f} do")
mn, mx = TK.doc_min_max()
bang(["Dac trung", "Gia tri goc", "min", "max", "Chuan hoa"],
     [[TR.TEN_COT[k], float(v[k]), float(mn[k]), float(mx[k]), float(xh["q"][k])] for k in range(7)])
o = np.argsort(-v[7:])[:4]
bang(["O mau (lon nhat)", "h_k goc", "Chuan hoa"], [[TR.TEN_COT[7 + k], float(v[7 + k]), float(xh["q"][7 + k])] for k in o])
bang(["Hang", "Anh", "D hinh hoc", "D ket cau", "D mau sac", "D", "S"],
     [[h + 1, xh["ds_ma"][i], float(xh["d_bo"]["hinh_hoc"][i]), float(xh["d_bo"]["ket_cau"][i]),
       float(xh["d_bo"]["mau_sac"][i]), float(xh["D"][i]), float(xh["S"][i])] for h, i in enumerate(xh["top"])])
print("S tren ca CSDL: nho nhat", round(float(xh["S"].min()), 4), "| trung vi", round(float(np.median(xh["S"])), 4))
hinh_ve, truc = plt.subplots(1, 6, figsize=(13, 2.9))
hien(truc[0], kq["rgb"], "Ảnh truy vấn")
for a, h, i in zip(truc[1:], range(1, 6), xh["top"]):
    hien(a, TR.doc_anh_da_xu_ly(xh["ds_ma"][i])[0], f"#{h} {xh['ds_ma'][i]}\nS = {xh['S'][i]:.4f}".replace(".", ","))
plt.tight_layout(); plt.savefig("figs/fig_truy_van_mau.png", dpi=130); plt.show()

### Đánh giá (mục 4c): phép thử ảnh biến đổi
`danh_gia.run()` mất khoảng 8 phút (tạo 8 biến thể cho mỗi ảnh, cho qua tiền xử lý và tìm kiếm), kết quả ở `data/danh_gia.csv`.

In [ ]:
# DG.run()        # bỏ dấu # để chạy lại phép thử
DG.tong_ket()

# Hình 4.x - 8 biến thể của ảnh mẫu
hinh_ve, truc = plt.subplots(1, 9, figsize=(16, 2.4))
goc = TXL.doc_rgb(os.path.join(XL.THU_MUC_ANH_GOC, MA_MAU + ".jpg"))
truc[0].imshow(goc); truc[0].set_title("ảnh gốc", fontsize=9); truc[0].axis("off")
for a, kieu in zip(truc[1:], DG.KIEU):
    a.imshow(DG.bien_doi(goc, kieu)[0]); a.set_title(DG.MO_TA[kieu], fontsize=7.5, wrap=True); a.axis("off")
plt.tight_layout(); plt.savefig("figs/fig_bien_the.png", dpi=130); plt.show()

### Top-5 của một số truy vấn (đánh giá bằng mắt)
Lấy ảnh trong CSDL làm truy vấn và không cho chính nó vào kết quả, coi như ảnh của một người chưa có trong CSDL.

In [ ]:
DS_TRUY_VAN = ["man_0050", "man_0153", "man_0300", "man_0500", "man_0700", "man_0900"]
hinh_ve, truc = plt.subplots(len(DS_TRUY_VAN), 6, figsize=(10, 1.95 * len(DS_TRUY_VAN)))
for hang, ma in enumerate(DS_TRUY_VAN):
    top = TK.tim_theo_vector(X[ds_ma.index(ma)], bo_qua=ma)
    hien(truc[hang, 0], TR.doc_anh_da_xu_ly(ma)[0], f"truy vấn {ma}")
    for a, r in zip(truc[hang, 1:], top):
        hien(a, TR.doc_anh_da_xu_ly(r["ma_anh"])[0], f"{r['ma_anh']}  S = {r['S']:.3f}".replace(".", ","))
plt.tight_layout(); plt.savefig("figs/fig_top5_nhieu.png", dpi=110); plt.show()